In [1]:
"""
File: train_emotion_classifier.py
Author: Octavio Arriaga
Email: arriaga.camargo@gmail.com
Github: https://github.com/oarriaga
Description: Train emotion classification model
"""

from keras.callbacks import CSVLogger, ModelCheckpoint, EarlyStopping
from keras.callbacks import ReduceLROnPlateau
from keras.preprocessing.image import ImageDataGenerator

from models.cnn import mini_XCEPTION
from utils.datasets_edited import DataManager
from utils.datasets_edited import split_data
# from utils.datasets_edited import
from utils.preprocessor import preprocess_input

from math import floor

import os
import numpy as np
import cv2
import pandas as pd

# parameters
batch_size = 32
# num_epochs = 10000
num_epochs = 2
input_shape = (64, 64, 1)
validation_split = .2
verbose = 1
num_classes = 7
patience = 50
base_path = '../trained_models/emotion_models/'

# data generator
data_generator = ImageDataGenerator(
                        featurewise_center=False,
                        featurewise_std_normalization=False,
                        rotation_range=10,
                        width_shift_range=0.1,
                        height_shift_range=0.1,
                        zoom_range=.1,
                        horizontal_flip=True)

Using TensorFlow backend.


In [26]:
# TRAIN DATA
img_shape = 64 # based on input image shape
split = .2 # test/val percentage
base_path = '../trained_models/emotion_models/'
# need variables declared in previous cell
# consider overriding modelcheckpoint functions to return last successfully saved model
def train(dataset_name, train_data, val_data, model):
    # pass dataset name
    # callbacks
    log_file_path = base_path + dataset_name + "_" + \
                    str(img_shape) +"_"+ str(split)+'_emotion_training.log'
    csv_logger = CSVLogger(log_file_path, append=False)
    early_stop = EarlyStopping('val_loss', patience=patience)
    reduce_lr = ReduceLROnPlateau('val_loss', factor=0.1,
                                patience=int(patience/4), verbose=1)
    trained_models_path = base_path + dataset_name + '_mini_XCEPTION'
    model_names = trained_models_path + '.{epoch:02d}-{val_acc:.2f}.hdf5'
    model_checkpoint = ModelCheckpoint(model_names, 'val_loss', verbose=1,
                                                    save_best_only=True)
    callbacks = [model_checkpoint, csv_logger, early_stop, reduce_lr]

    # loading dataset
    faces, emotions = train_data
    temp = model.fit_generator(data_generator.flow(faces, emotions,
                                            batch_size),
                        steps_per_epoch=len(faces) / batch_size,
                        epochs=num_epochs, verbose=1, callbacks=callbacks,
                        validation_data=val_data)
    
    return temp
    ### TRAINING

In [ ]:
# TEST DATA
def test():
    # load model
    # "unload" model

In [3]:
# count labels from 1d array of labels
# input: labels int array || num_classes int
# output: array of instances count (size according to num_classes)
num_classes = 7
def count_labels(labels_int):
    labels_count = [0 for x in range(num_classes)]
    for label_int in labels_int:
        labels_count[label_int]+=1
    return labels_count

In [16]:
# get first n instances per label from labels array, get faces array of same index
# does not randomize because assumes input is already shuffled
# outputs an array of numpy arrays (according to class)
# input: n, faces, labels
# output: faces_subset, labels_subset
num_classes = 7
def get_subsets(minimum, faces, emotions):
    counter = [minimum for x in range(num_classes)]
    subsets_faces = [[] for x in range(num_classes)]
    subsets_emotions = [[] for x in range(num_classes)]
    
    i = 0
    while sum(counter) != 0:
        label_int = labels_int[i]
        if counter[label_int] != 0:
            if subsets_faces[label_int] == []:
                temp_face = np.expand_dims(faces[i], axis=0)
                temp_emotion = np.expand_dims(emotions[i], axis=0)
                subsets_faces[label_int] = temp_face
                subsets_emotions[label_int] = temp_emotion
            else:
                temp_face = np.expand_dims(faces[label_int], axis=0)
                temp_emotion = np.expand_dims(emotions[label_int], axis=0)
                subsets_faces[label_int] = np.vstack((subsets_faces[label_int],\
                                                            temp_face))
                subsets_emotions[label_int] = np.vstack((subsets_emotions[label_int],\
                                                               temp_emotion))
            counter[label_int] -= 1
        i += 1
    
    return subsets_faces, subsets_emotions

In [19]:
# split data to train, test, and val according and percentage
# input: faces_subset, labels_subset, split_percentage
# output: train_data, val_data, test_data 
def split_data(x, y, split):
    num_samples = len(x)
    num_val_samples = int(round(split*num_samples))
    num_train_samples = int(round((1 - split*2)*num_samples))
#     print((1-split*2)*num_samples)
#     print(split*num_samples)
    train_x = x[:num_train_samples]
    train_y = y[:num_train_samples]
#     print("---------train size: "+str(len(train_x)))
    val_x = x[num_train_samples:num_train_samples+num_val_samples]
    val_y = y[num_train_samples:num_train_samples+num_val_samples]
#     print("---------val size: "+str(len(val_x)))
    test_x = x[num_train_samples+num_val_samples:]
    test_y = y[num_train_samples+num_val_samples:]
#     print("---------test size: "+str(len(test_x)))
    train_data = (train_x, train_y)
    val_data = (val_x, val_y)
    test_data = (test_x, test_y)
    return train_data, val_data, test_data


# input: faces array (array of numpy arrays),
#        emotions array (array of numpy arrays),
#        split int percentage of train/test data
# output: train_data - tuple of two numpy arrays
#         val_data - tuple of two numpy arrays
#         test_data - tuple of two numpy arrays
num_classes = 7
def strat_sample(faces, emotions, split):
    for i in range(num_classes):
        # split according to class
        temp_train, temp_val, temp_test = split_data(faces[i],
                                                     emotions[i],
                                                     split)
        if i == 0:
            train_faces, train_emotions = temp_train
            val_faces, val_emotions = temp_val
            test_faces, test_emotions = temp_test
        else:
            train_faces = np.vstack((train_faces,
                                    temp_train[0]))
            train_emotions = np.vstack((train_emotions,
                                       temp_train[1]))
            val_faces = np.vstack((val_faces,
                                  temp_val[0]))
            val_emotions = np.vstack((val_emotions,
                                     temp_val[1]))
            test_faces = np.vstack((test_faces,
                                   temp_test[0]))
            test_emotions = np.vstack((test_emotions,
                                      temp_test[1]))
    
    train_data = (train_faces, train_emotions)
    val_data = (val_faces, val_emotions)
    test_data = (test_faces, test_emotions)
    
    return train_data, val_data, test_data

In [5]:
# LOAD IMAGES AND LABELS IF CSV EXISTS
def load_CSV(csv_file, img_size):
    data = pd.read_csv(csv_file)
    pixels = data['face'].tolist()
    # width, height = 64, 64
    faces = []
    for pixel_sequence in pixels:
        face = [int(pixel) for pixel in pixel_sequence.split(' ')]
        face = np.asarray(face).reshape(img_size[0], img_size[1])
        face = cv2.resize(face.astype('uint8'), img_size)
        faces.append(face.astype('float32'))
    faces = np.asarray(faces)
    faces = np.expand_dims(faces, -1)
    emotions = pd.get_dummies(data['emotion']).as_matrix()

    return faces, emotions

In [6]:
# SHUFFLE FACES AND EMOTION ARRAYS SIMULTANEOUSLY
# source: https://stackoverflow.com/questions/4601373/better-way-to-shuffle-two-numpy-arrays-in-unison
def shuffle_in_unison(a, b):
    assert len(a) == len(b)
    shuffled_a = np.empty(a.shape, dtype=a.dtype)
    shuffled_b = np.empty(b.shape, dtype=b.dtype)
    permutation = np.random.permutation(len(a))
    for old_index, new_index in enumerate(permutation):
        shuffled_a[new_index] = a[old_index]
        shuffled_b[new_index] = b[old_index]
    return (shuffled_a, shuffled_b)

In [7]:
def get_labels(dataset_name):
    if dataset_name in ['FER2013', 'ExpW', 'ExpWCSV']:
        return {0: 'angry', 1: 'disgust', 2: 'fear', 3: 'happy',
                4: 'sad', 5: 'surprise', 6: 'neutral'}
    elif dataset_name == 'imdb':
        return {0: 'woman', 1: 'man'}
    elif dataset_name in ['KDEF', 'KDEFCSV']:
        return {0: 'AN', 1: 'DI', 2: 'AF', 3: 'HA', 4: 'SA', 5: 'SU', 6: 'NE'}
    elif dataset_name in ['AffectNet', 'AffectNetCSV']:
        return {0: 'neutral', 1: 'happy', 2: 'sad', 3: 'surprise', 
                4: 'fear', 5: 'disgust', 6: 'anger'}
    else:
        raise Exception('Invalid dataset name')


def get_class_to_arg(dataset_name='FER2013'):
    if dataset_name in ['FER2013', 'ExpW', 'ExpWCSV']:
        return {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4,
                'surprise': 5, 'neutral': 6}
    elif dataset_name == 'imdb':
        return {'woman': 0, 'man': 1}
    elif dataset_name in ['KDEF', 'KDEFCSV']:
        return {'AN': 0, 'DI': 1, 'AF': 2, 'HA': 3, 'SA': 4, 'SU': 5, 'NE': 6}
    elif dataset_name in ['AffectNet', 'AffectNetCSV']:
        return {'neutral': 0, 'happy': 1, 'sad': 2, 'surprise': 3, 
                'fear': 4, 'disgust': 5, 'anger': 6}
    else:
        raise Exception('Invalid dataset name')

In [8]:
# DATASETS TO BE USED
datasets_root = "/home/ubuntu/Datasets"
# datasets = ['AffectNet', 'FER2013', 'KDEF', 'ExpW']
datasets = ['KDEF']#, 'ExpW']
input_shape = (64, 64, 1)
save_to_csv = []
split = 0.1
# csv_output = "collected_datasets.csv"

dataset_dict = {}

In [9]:
# LOAD DATASETS INTO DICTIONARY
# ----dictionary structure: {<dataset_name>:[faces, emotions, model_created]}
print("Loading datasets...")
for i, dataset in enumerate(datasets):
    # load/create csv
    csv_name = str(dataset) + "_" + str(input_shape[0]) + ".csv"
    try:
        csv_file = open(os.path.join(datasets_root, dataset, csv_name))
    except:
        print("["+str(dataset)+"] File "+csv_name+" does not exist. Loading and converting dataset.")
        path = os.path.join(datasets_root, dataset)
        csv_path = os.path.join(datasets_root, dataset, csv_name)
        data_loader = DataManager(dataset,
                                  image_size=input_shape[:2],
                                  dataset_path=path,
                                  csv_output=csv_path)
        faces, emotions = data_loader.get_data()
    else:
        print("["+str(dataset)+"] File "+csv_name+" exists. Loading file...")
        faces, emotions = load_CSV(csv_file, input_shape[:2])

    # shuffle
    print("["+str(dataset)+"] Shuffling images and labels simultaneously...")
    (faces, emotions) = shuffle_in_unison(faces,
                                        emotions)
    print("["+str(dataset)+"] Creating miniXception model...")
    dataset_dict[dataset] = [faces,
                             emotions,
                             mini_XCEPTION(input_shape, num_classes)]
    
print("Finished loading datasets.\n")

Loading datasets...
[KDEF] File KDEF_64.csv exists. Loading file...


/home/ubuntu/miniconda3/envs/capstone_2/lib/python3.6/site-packages/ipykernel_launcher.py:14: FutureWarning: Method .as_matrix will be removed in a future version. Use .values instead.
  


[KDEF] Shuffling images and labels simultaneously...
[KDEF] Creating miniXception model...
Finished loading datasets.



In [27]:
# TRAINING AND TESTING SECTION
for i, dataset in enumerate(datasets):
    # preprocess all loaded images
    print("-------------------------------------------")
    print("["+str(dataset)+"] Preprocessing dataset...")
#     faces = preprocess_input(dataset_dict[dataset][0])
    faces = dataset_dict[dataset][0]
    #     print("["+str(dataset)+"] Acquiring labels...")
    emotions = dataset_dict[dataset][1]
    labels = get_labels(dataset)
    labels_int = []
    num_samples, num_classes = emotions.shape

    print("["+str(dataset)+"] Counting class instances...")
    
    # generate array for counting label instances
    labels_int = [emotion.argmax() for emotion in emotions]
    labels_count = count_labels(labels_int)
    minimum = min(labels_count)
    min_index = labels_count.index(minimum)
    
    
    # print class per line
    print("["+str(dataset)+"] (LABEL) count:")
    for i in range(num_classes):
        temp = "["+str(dataset)+"]  ("+str(labels[i])+") "+str(labels_count[i])
        if i == min_index:
            temp = temp+" <--- minimum"
        print(temp)
        
    # get minimum samples from datasets
    print("["+str(dataset)+"] Acquiring subsets with "+str(minimum)+" samples from each class...")
    subsets_faces, subsets_emotions = get_subsets(minimum, faces, emotions)
    print("["+str(dataset)+"] Acquired images subset with size "+str(len(subsets_faces[0]))+" per class.")
    print("["+str(dataset)+"] Acquired labels subset with size "+str(len(subsets_emotions[0]))+" per class.")    
    
    
    # split data
    train_count = round(minimum * (1-split*2))
    val_count = round(minimum * split)
    test_count = round(minimum * split)
    print("["+str(dataset)+"] Splitting subsets with "+\
          str((1-split*2)*100)+"|"+\
          str(split*100)+"|"+\
          str(split*100)+" split...")
    print("["+str(dataset)+"] Split would be "+\
          str(train_count)+"|"+\
          str(val_count)+"|"+\
          str(test_count)+" per class.")
    
#   split data and join data of different classes in a single 4d numpy array
    train_data, test_data, val_data = strat_sample(subsets_faces,
                                                   subsets_emotions,
                                                   split)

    print("["+str(dataset)+"] (Train data) size: "+\
          str(train_data[0].shape[0]))
    print("["+str(dataset)+"] (Val  data) size: "+\
          str(val_data[0].shape[0]))
    print("["+str(dataset)+"] (Test  data) size: "+\
          str(test_data[0].shape[0]))

    # shuffle training and val data
    print("["+str(dataset)+"] Shuffling (train) images and labels simultaneously...")
    train_data = shuffle_in_unison(train_data[0],
                                   train_data[1])
    print("["+str(dataset)+"] Shuffling ( val ) images and labels simultaneously...")
    val_data = shuffle_in_unison(val_data[0],
                                 val_data[1])
    

    # compile model, model is stored in dict
    print("["+str(dataset)+"] Compiling model for training...")
    dataset_dict[dataset][2].compile(optimizer='adam', loss='categorical_crossentropy',
                metrics=['accuracy'])

    # train model, log saved, see function
    print("["+str(dataset)+"] Training...")
    model_file = train(dataset, train_data[:100], val_data[:20], dataset_dict[dataset][2])
    print("yoyoyo "+model_file)
#     test()
#     # log() actual and predicted outputs

#     for j, test_dataset in enumerate(datasets):
#         if j == i:
#             continue
#         # is trained model loaded at this point?
#         # for images in dataset_dict[test_dataset]
#         # test on model
#         # log actual and predicted outputs in <train_dataset>_<test_dataset>

-------------------------------------------
[KDEF] Preprocessing dataset...
[KDEF] Counting class instances...
[KDEF] (LABEL) count:
[KDEF]  (AN) 700
[KDEF]  (DI) 700
[KDEF]  (AF) 700
[KDEF]  (HA) 700
[KDEF]  (SA) 699 <--- minimum
[KDEF]  (SU) 699
[KDEF]  (NE) 700
[KDEF] Acquiring subsets with 699 samples from each class...
[KDEF] Acquired images subset with size 699 per class.
[KDEF] Acquired labels subset with size 699 per class.
[KDEF] Splitting subsets with 60.0|20.0|20.0 split...
[KDEF] Split would be 419|140|140 per class.
[KDEF] (Train data) size: 2933
[KDEF] (Val  data) size: 980
[KDEF] (Test  data) size: 980
[KDEF] Shuffling (train) images and labels simultaneously...
[KDEF] Shuffling ( val ) images and labels simultaneously...
[KDEF] Compiling model for training...
[KDEF] Training...
Epoch 1/2
92/91 [==============================] - 7s 80ms/step - loss: 0.0268 - acc: 0.9980 - val_loss: 11.5219 - val_acc: 0.2857

Epoch 00001: val_loss improved from inf to 11.52188, saving mod

TypeError: must be str, not History

In [28]:
model_file

In [ ]:
print(faces.shape)
print(daces[0].shape)
# subsets_faces
# train_faces[label_indexes[i]].shape
print(subsets_faces[0].shape)
print(subsets_faces[0][0].shape)

In [ ]:
from keras.models import load_model

# TESTING SECTION
# for i, dataset in enumerate(datasets):
# load model
# open file
# for all data inputted to model
# get classification
# print actual then predicted on file
# close filie
fer_path = "/home/ubuntu/Repos/face_classification/trained_models/undersampled"
# fer_path = "/home/ubuntu/Repos/face_classification/trained_models/emotion_models/default"
# 'AffectNet', 'FER2013', 'KDEF', 'ExpW'
# fer_models = ["AffectNet_mini_XCEPTION.100-1.00.hdf5",
#               "FER2013_mini_XCEPTION.118-1.00.hdf5",
#               "KDEF_mini_XCEPTION.158-1.00.hdf5",
#               "ExpW_mini_XCEPTION.36-1.00.hdf5"]
fer_models = ["KDEF_mini_XCEPTION.158-1.00.hdf5",
              "ExpW_mini_XCEPTION.36-1.00.hdf5"]
# fer_models = ["ALL_mini_XCEPTION.143-0.72.hdf5",
#               "fer2013_mini_XCEPTION.110-0.65.hdf5"]
# model_name = ""
for i, fer_model in enumerate(fer_models):
    print(fer_model)
    emotion_model_path = os.path.join(fer_path, fer_model)
    emotion_labels = get_labels(datasets[i])
    # Load models

    emotion_classifier = load_model(emotion_model_path, compile=False)
    emotion_target_size = emotion_classifier.input_shape[1:3]

    for j, test_dataset in enumerate(datasets):
        if j == i:
            continue
            
        f = open(datasets[i]+"_"+datasets[j]+".txt", 'w')
        
        for k, image in enumerate(dataset_dict[datasets[j]][0]):    
            
            try:
                gray_face = cv2.resize(image, (emotion_target_size))
            except:
                continue
            
            gray_face = preprocess_input(gray_face, True)
            gray_face = np.expand_dims(gray_face, 0)
            gray_face = np.expand_dims(gray_face, -1)

            emotion_prediction = emotion_classifier.predict(gray_face)
            emotion_probability = np.max(emotion_prediction)  # returns probability value
            emotion_label_arg = np.argmax(emotion_prediction)  # returns index of highest probability value
            emotion_text = emotion_labels[emotion_label_arg]
            
            actual_label = int(np.argwhere(dataset_dict[datasets[j]][1][k]==1))
            pred_label = emotion_label_arg
#             f.write(str(pred_label)+","+str(actual_label)+"\n")
            print(str(pred_label)+","+str(actual_label))
        f.close
    del(dataset_dict[dataset[i]][2])
    gc.collect()